# Laboratoire 12 : Benchmark multi-plateforme

**Semaine 13 — Cours de Quantum Machine Learning (Master/PhD)**

## Objectifs

- Implémenter un même circuit VQC dans trois frameworks : PennyLane, Qiskit, Cirq
- Comparer les résultats sur simulateur
- Intégrer Amazon Braket : soumission sur simulateurs IonQ/Rigetti
- Analyser le bruit et la calibration
- Formuler des recommandations : choix de plateforme selon la tâche

---
## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.figsize': (8, 5)})
print(f"NumPy   : {np.__version__}")

---
## Partie A : Même circuit VQC dans trois frameworks

Nous définissons un circuit variationnel simple :
- Encodage par angle (4 features)
- 2 couches variationnelles (RX + RY + CNOT)
- Mesure de l'espérance de $Z_0$

### A.1 PennyLane

In [ ]:
import pennylane as qml

# Circuit VQC PennyLane
def circuit_pennylane(params, x):
    qml.AngleEmbedding(x, wires=range(4))
    for l in range(2):
        for i in range(4):
            qml.RX(params[l, i, 0], wires=i)
            qml.RY(params[l, i, 1], wires=i)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[2, 3])
    return qml.expval(qml.PauliZ(0))

dev_pl = qml.device("default.qubit", wires=4)
qnode_pl = qml.QNode(circuit_pennylane, dev_pl)

np.random.seed(42)
params_pl = np.random.uniform(-np.pi, np.pi, size=(2, 4, 2))
x_test = np.array([0.5, -0.3, 0.8, 0.1])

result_pl = qnode_pl(params_pl, x_test)
print(f"PennyLane : {result_pl:.6f}")

### A.2 Qiskit

In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit.circuit import Parameter
    from qiskit.primitives import Estimator
    from qiskit.quantum_info import SparsePauliOp
    
    HAS_QISKIT = True
    print(f"Qiskit disponible.")
except ImportError:
    HAS_QISKIT = False
    print("Qiskit non disponible. Installez avec : pip install qiskit qiskit-aer")

In [ ]:
if HAS_QISKIT:
    # Construction du circuit Qiskit
    qc = QuantumCircuit(4)
    
    # Encodage (paramètres de données)
    x_params = [Parameter(f"x{i}") for i in range(4)]
    for i, p in enumerate(x_params):
        qc.rx(p, i)
    
    # Couches variationnelles
    theta = []
    for l in range(2):
        for i in range(4):
            rx_p = Parameter(f"th_{l}_{i}_rx")
            ry_p = Parameter(f"th_{l}_{i}_ry")
            theta.extend([rx_p, ry_p])
            qc.rx(rx_p, i)
            qc.ry(ry_p, i)
        qc.cx(0, 1)
        qc.cx(2, 3)
    
    # Observable
    obs = SparsePauliOp.from_list([("ZZZZ", 1.0)])
    
    # Estimator
    estimator = Estimator()
    
    # Binding des paramètres
    param_values = {}
    for i, p in enumerate(x_params):
        param_values[p] = x_test[i]
    idx = 0
    for l in range(2):
        for i in range(4):
            param_values[theta[idx]] = float(params_pl[l, i, 0])
            param_values[theta[idx + 1]] = float(params_pl[l, i, 1])
            idx += 2
    
    job = estimator.run([qc], [obs], [param_values])
    result_qk = job.result().values[0]
    print(f"Qiskit    : {result_qk:.6f}")

### A.3 Cirq

In [ ]:
try:
    import cirq
    HAS_CIRQ = True
    print(f"Cirq disponible.")
except ImportError:
    HAS_CIRQ = False
    print("Cirq non disponible. Installez avec : pip install cirq")

In [ ]:
if HAS_CIRQ:
    # Circuit VQC Cirq
    qubits = cirq.LineQubit.range(4)
    
    def vqc_cirq(params, x):
        circuit = cirq.Circuit()
        # Encodage
        for i, val in enumerate(x):
            circuit.append(cirq.rx(val).on(qubits[i]))
        # Couches
        for l in range(2):
            for i in range(4):
                circuit.append(cirq.rx(params[l, i, 0]).on(qubits[i]))
                circuit.append(cirq.ry(params[l, i, 1]).on(qubits[i]))
            circuit.append(cirq.CNOT(qubits[0], qubits[1]))
            circuit.append(cirq.CNOT(qubits[2], qubits[3]))
        return circuit
    
    circuit_cq = vqc_cirq(params_pl, x_test)
    
    # Simulation avec mesure de Pauli Z sur qubit 0
    sim = cirq.Simulator()
    # Valeur d'espérance via simulation d'état
    result_state = sim.simulate(circuit_cq)
    state_vector = result_state.final_state_vector
    
    # Calcul de <Z0> = <psi|Z0|psi>
    n = 4
    expval = 0.0
    for i, amp in enumerate(state_vector):
        # Z0 donne +1 si bit 0 = 0, -1 si bit 0 = 1
        z = 1 if (i >> (n - 1 - 0)) & 1 == 0 else -1
        expval += z * np.abs(amp) ** 2
    
    print(f"Cirq      : {expval:.6f}")

---
## Partie B : Comparaison des résultats

Nous comparons numériquement les résultats des trois frameworks pour le même circuit et les mêmes paramètres.

In [ ]:
results = {"PennyLane": result_pl}
if HAS_QISKIT:
    results["Qiskit"] = result_qk
if HAS_CIRQ:
    results["Cirq"] = expval

print("Comparaison des résultats :")
print("-" * 40)
for name, val in results.items():
    print(f"  {name:12s} : {val:.8f}")

if len(results) >= 2:
    ref = list(results.values())[0]
    max_diff = max(abs(v - ref) for v in results.values())
    print(f"\nÉcart maximal entre frameworks : {max_diff:.2e}")
    if max_diff < 1e-10:
        print("✓ Les résultats coïncident parfaitement.")
    else:
        print("⚠ De légers écarts peuvent venir des différences de simulateur.")

---
## Partie C : Intégration Amazon Braket

Amazon Braket donne accès à des simulateurs et matériels quantiques (IonQ, Rigetti, QuEra) via une API unique.

**Note** : L'exécution sur Braket nécessite des credentials AWS. Cette section utilise le simulateur local par défaut.

In [ ]:
try:
    from braket.aws import AwsDevice
    from braket.devices import LocalSimulator
    from braket.circuits import Circuit, Observable
    
    HAS_BRAKET = True
    print("Amazon Braket SDK disponible.")
    
    # Simulateur local Braket
    braket_sim = LocalSimulator()
    print(f"Backends disponibles : {braket_sim.properties}"[:200])
except ImportError as e:
    HAS_BRAKET = False
    print(f"Amazon Braket SDK non installé : {e}")
    print("Installez avec : pip install amazon-braket-sdk")

In [ ]:
if HAS_BRAKET:
    # Construction du circuit Braket
    braket_circuit = Circuit()
    
    # Encodage
    for i, val in enumerate(x_test):
        braket_circuit.rx(i, val)
    
    # Couches variationnelles
    for l in range(2):
        for i in range(4):
            braket_circuit.rx(i, float(params_pl[l, i, 0]))
            braket_circuit.ry(i, float(params_pl[l, i, 1]))
        braket_circuit.cnot(0, 1)
        braket_circuit.cnot(2, 3)
    
    # Mesure de l'espérance de Z sur le qubit 0
    braket_circuit.expectation(observable=Observable.Z(), target=0)
    
    # Exécution
    task = braket_sim.run(braket_circuit, shots=0)  # shots=0 -> calcul exact
    result_braket = task.result()
    
    print(f"Amazon Braket (simulateur local) : {result_braket.values[0]:.6f}")
    
    # Comparaison
    print(f"Différence avec PennyLane : {abs(result_braket.values[0] - result_pl):.2e}")

### Soumission sur simulateur IonQ/Rigetti (commenté)

Pour soumettre sur du matériel réel (nécessite credentials AWS) :

```python
# ionq = AwsDevice("arn:aws:braket:us-east-1::device/qpu/ionq/Aria-1")
# rigetti = AwsDevice("arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-2")
# task_ionq = ionq.run(braket_circuit, shots=1000)
# result_ionq = task_ionq.result()
```

---
## Partie D : Analyse du bruit et calibration

Nous simulons un bruit réaliste pour étudier son impact sur le VQC.

In [ ]:
# Modèle de bruit dépolarisant avec PennyLane
def noisy_vqc(params, x, noise_level=0.01):
    dev_noisy = qml.device("default.mixed", wires=4)
    
    @qml.qnode(dev_noisy)
    def circuit():
        qml.AngleEmbedding(x, wires=range(4))
        for l in range(2):
            for i in range(4):
                qml.RX(params[l, i, 0], wires=i)
                qml.RY(params[l, i, 1], wires=i)
            qml.CNOT(wires=[0, 1])
            qml.CNOT(wires=[2, 3])
            # Bruit dépolarisant après chaque couche
            for i in range(4):
                qml.DepolarizingChannel(noise_level, wires=i)
        return qml.expval(qml.PauliZ(0))
    
    return circuit()

# Test avec différents niveaux de bruit
noise_levels = [0.0, 0.001, 0.01, 0.05, 0.1]
results_noisy = []

for nlev in noise_levels:
    val = noisy_vqc(params_pl, x_test, noise_level=nlev)
    results_noisy.append(val)
    print(f"Bruit {nlev:.3f} : <Z> = {val:.6f}")

In [ ]:
plt.figure()
plt.semilogx(noise_levels, results_noisy, "bo-", lw=2, markersize=8)
plt.axhline(result_pl, color="gray", ls="--", label="Sans bruit")
plt.xlabel("Niveau de bruit dépolarisant")
plt.ylabel("<Z>")
plt.title("Impact du bruit sur la mesure du VQC")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nDéviation à bruit=0.1 : {abs(results_noisy[-1] - result_pl):.4f}")
print("Le bruit dépolarisant fait tendre <Z> vers 0 (état maximale mixte).")

### Calibration et métriques matérielles

Voici les spécifications typiques des plateformes actuelles (données 2026) :

In [ ]:
platforms = {
    "IBM Brisbane": {
        "type": "Supraconducteur",
        "qubits": 127,
        "gate_error": 3e-3,
        "readout_error": 5e-2,
        "T1 (us)": 250,
        "T2 (us)": 150,
        "connectivité": "Heavy-hex"
    },
    "IonQ Aria-1": {
        "type": "Ions piégés",
        "qubits": 25,
        "gate_error": 1e-4,
        "readout_error": 1e-2,
        "T1 (us)": 10000,
        "T2 (us)": 5000,
        "connectivité": "All-to-all"
    },
    "Rigetti Ankaa-2": {
        "type": "Supraconducteur",
        "qubits": 84,
        "gate_error": 5e-3,
        "readout_error": 8e-2,
        "T1 (us)": 50,
        "T2 (us)": 35,
        "connectivité": "Planar"
    },
    "Google Willow": {
        "type": "Supraconducteur",
        "qubits": 105,
        "gate_error": 1e-3,
        "readout_error": 3e-2,
        "T1 (us)": 100,
        "T2 (us)": 90,
        "connectivité": "Planar"
    }
}

print(f"{'Plateforme':18s} | {'Type':15s} | {'Qubits':6s} | {'Gate err':8s} | {'Readout':8s} | {'T2 (us)':8s}")
print("-" * 80)
for name, spec in platforms.items():
    print(f"{name:18s} | {spec['type']:15s} | {spec['qubits']:6d} | {spec['gate_error']:.0e}  | {spec['readout_error']:.0e}  | {spec['T2 (us)']:8.0f}")

---
## Partie E : Recommandations — choix de plateforme selon la tâche

### Arbre de décision

| Tâche | Plateforme recommandée | Justification |
|-------|----------------------|--------------|
| Chimie quantique (VQE, < 30 qubits) | PennyLane + IBM Brisbane | Ansätze UCCSD natifs, couche chimique intégrée |
| Kernels quantiques (grande échelle) | Qiskit ML + IBM Quantum | FidelityKernel optimisé, accès 127 qubits |
| Optimisation (QAOA, < 30 qubits) | PennyLane + IonQ | Haute fidélité, connectivité complète |
| Classification (VQC, < 15 qubits) | PennyLane / Cirq + tout simulateur | Faible besoin en qubits, flexibilité |
| Prototypage rapide | PennyLane (simulateur local) | Différentiation automatique, intégration ML |
| Recherche (barren plateaux, Fisher) | Cirq + Stim | Simulation stabilisatrice, optimisation NISQ |
| Déploiement industriel | Amazon Braket (multi-backend) | API unifiée, scaling automatique |
| Multi-framework reproductible | ONNX Export [Fra26] | Interopérabilité, export vers tous les backends |

In [ ]:
# Résumé : score composite par plateforme
scores = {
    "IBM Brisbane": {"fidelite": 7, "echelle": 10, "connectivite": 6, "ecosysteme": 9},
    "IonQ Aria-1": {"fidelite": 10, "echelle": 5, "connectivite": 10, "ecosysteme": 6},
    "Rigetti Ankaa-2": {"fidelite": 6, "echelle": 8, "connectivite": 7, "ecosysteme": 5},
    "Google Willow": {"fidelite": 8, "echelle": 9, "connectivite": 7, "ecosysteme": 7},
}

categories = ["fidelite", "echelle", "connectivite", "ecosysteme"]
x = np.arange(len(categories))
width = 0.2

plt.figure(figsize=(10, 5))
for i, (name, score) in enumerate(scores.items()):
    vals = [score[c] for c in categories]
    plt.bar(x + i * width, vals, width, label=name)

plt.xlabel("Critère")
plt.ylabel("Score (0-10)")
plt.title("Comparaison multi-critères des plateformes QML")
plt.xticks(x + width * 1.5, categories)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Exercices — Pour aller plus loin

1. **Benchmark quantitatif** : Lancez 100 circuits aléatoires sur chaque simulateur et comparez les temps d'exécution.
2. **Transpilation** : Transpilez le circuit VQC pour l'architecture IBM Brisbane (heavy-hex) avec Qiskit. Combien de portes SWAP sont ajoutées ?
3. **Bruit réaliste** : Utilisez `qml.NoiseModel` de PennyLane ou les modèles de bruit Qiskit Aer pour simuler IBM Brisbane.
4. **Error mitigation** : Implémentez la zero-noise extrapolation (ZNE) avec `mitiq` pour corriger le bruit.
5. **Braket réel** : Si vous avez un compte AWS, soumettez le circuit sur le simulateur IonQ (`ionq.simulator`) et comparez.
6. **Export ONNX** : Utilisez [Fra26] pour exporter le circuit au format ONNX et le recharger sur une autre plateforme.